# 01: a dense-network baseline on the artworks (milestone m1)

WikiArt subset: 5 genres (cityscape, landscape, portrait, religious_painting, still_life), 300 images per
genre, 128 x 128 pixels. Built with `make_wikiart_subset.py` (exact command in the README).

## Imports

- `torch`, `torch.nn`, `torch.utils.data`: PyTorch, as in Session 4.
- `torchvision.datasets`: `ImageFolder` reads the subset (one folder per genre).
- `sklearn.metrics.confusion_matrix`: counts, for each true class, how often each class was predicted.
- `pandas`: reads `metadata.csv` (the artist of each image).
- `numpy`, `matplotlib`: arrays and plots.

In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets
from sklearn.metrics import confusion_matrix

DATA_DIR = "data/wikiart_genre5"
np.random.seed(0)
torch.manual_seed(0)
torch.set_num_threads(2)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## Helper functions

`evaluate`, `train_one_epoch` and `fit` are the functions of TP04, as given at the top of TP05 (the clean training
loop of Session 4). `fit` returns a dictionary with one value per epoch for `"train_loss"`, `"val_loss"` and
`"val_acc"`.

In [2]:
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total


def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    running, total = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item() * len(yb)
        total += len(yb)
    return running / total


def fit(model, train_loader, val_loader, loss_fn, optimizer, n_epochs, verbose=True):
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(n_epochs):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if verbose:
            print(f"epoch {epoch + 1:2d} | train loss {train_loss:.4f} | val loss {val_loss:.4f} | val acc {val_acc:.3f}")
    return history